In [1]:
import sys
sys.path.append('..')

In [2]:
from src.modeling import ProjectionVIT, MLP
from src.data import inference_examination_to_tensor

/home/borntowarn/projects/chest-diseases/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/borntowarn/projects/chest-diseases/venv/lib/python3.11/site-packages/vector_quantize_pytorch/vector_quantize_pytorch.py:261: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/home/borntowarn/projects/chest-diseases/venv/lib/python3.11/site-packages/vector_quantize_pytorch/vector_quantize_pytorch.py:391: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


In [3]:
import torch

model_lipro = ProjectionVIT()
model_lipro.load_state_dict(
    torch.load('../weights/CT-RATE/ProjectionVIT_LiPro_V2.pt')
)
model_lipro.cuda()

ProjectionVIT(
  (VIT): CTViT(
    (spatial_rel_pos_bias): ContinuousPositionBias(
      (net): ModuleList(
        (0): Sequential(
          (0): Linear(in_features=2, out_features=512, bias=True)
          (1): LeakyReLU(negative_slope=0.1)
        )
        (1): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=True)
          (1): LeakyReLU(negative_slope=0.1)
        )
        (2): Linear(in_features=512, out_features=8, bias=True)
      )
    )
    (to_patch_emb_first_frame): Sequential(
      (0): Rearrange('b c 1 (h p1) (w p2) -> b 1 h w (c p1 p2)', p1=20, p2=20)
      (1): LayerNorm((400,), eps=1e-05, elementwise_affine=True)
      (2): Linear(in_features=400, out_features=512, bias=True)
      (3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (to_patch_emb): Sequential(
      (0): Rearrange('b c (t pt) (h p1) (w p2) -> b t h w (c pt p1 p2)', p1=20, p2=20, pt=10)
      (1): LayerNorm((4000,), eps=1e-05, elementwise_affine=True)
     

In [4]:
# Вручную перевести нужные слои в режим eval()

# Преобразуем все BatchNorm и Dropout уровни в eval без перевода всей модели
def set_all_bn_dropout_eval(module):
    """
    Рекурсивно переводит все BatchNorm* и Dropout* модули в режим eval.
    """
    for submodule in module.children():
        # BatchNorm
        if isinstance(submodule, torch.nn.modules.batchnorm._BatchNorm):
            submodule.eval()
        # Dropouts
        if isinstance(submodule, torch.nn.Dropout) or isinstance(submodule, torch.nn.Dropout2d) or isinstance(submodule, torch.nn.Dropout3d):
            submodule.eval()
        set_all_bn_dropout_eval(submodule)

set_all_bn_dropout_eval(model_lipro)


In [5]:
for name, module in model_lipro.named_modules():
    if module.training is False:
        print(f"{name}: {module.__class__.__name__} in eval mode")


VIT.enc_spatial_transformer.layers.0.1.attn_dropout: Dropout in eval mode
VIT.enc_spatial_transformer.layers.0.3.3: Dropout in eval mode
VIT.enc_spatial_transformer.layers.1.1.attn_dropout: Dropout in eval mode
VIT.enc_spatial_transformer.layers.1.3.3: Dropout in eval mode
VIT.enc_spatial_transformer.layers.2.1.attn_dropout: Dropout in eval mode
VIT.enc_spatial_transformer.layers.2.3.3: Dropout in eval mode
VIT.enc_spatial_transformer.layers.3.1.attn_dropout: Dropout in eval mode
VIT.enc_spatial_transformer.layers.3.3.3: Dropout in eval mode
VIT.enc_temporal_transformer.layers.0.1.attn_dropout: Dropout in eval mode
VIT.enc_temporal_transformer.layers.0.3.3: Dropout in eval mode
VIT.enc_temporal_transformer.layers.1.1.attn_dropout: Dropout in eval mode
VIT.enc_temporal_transformer.layers.1.3.3: Dropout in eval mode
VIT.enc_temporal_transformer.layers.2.1.attn_dropout: Dropout in eval mode
VIT.enc_temporal_transformer.layers.2.3.3: Dropout in eval mode
VIT.enc_temporal_transformer.layers

In [6]:
# input_tensor = inference_examination_to_tensor('/home/borntowarn/projects/chest-diseases/training/data/CT-RATE/dataset/valid_fixed/valid_16/valid_16_a/valid_16_a_1.nii.gz')
input_tensor = inference_examination_to_tensor('/home/borntowarn/projects/chest-diseases/training/data/CT-RATE/dataset/train_fixed/train_19/train_19_a/train_19_a_1.nii.gz')

2025-10-16 10:54:27.990 | INFO     | src.data:inference_examination_to_tensor:327 - Обработка NIFTI исследования
2025-10-16 10:54:27.991 | INFO     | src.data:inference_examination_to_tensor:330 - Чтение финальной серии из /home/borntowarn/projects/chest-diseases/training/data/CT-RATE/dataset/train_fixed/train_19/train_19_a/train_19_a_1.nii.gz
2025-10-16 10:54:30.174 | INFO     | src.data:inference_examination_to_tensor:353 - Slope: 1, Intercept: 0


In [7]:
binary_model = MLP(
    input_size=512,
    activation="gelu",
    dropout=0.2,
    hidden_sizes=[256, 128],
    num_classes=2,
)
binary_model.load_state_dict(torch.load('../weights/CT-RATE/model_binary.pth'))
binary_model.eval().cuda()


MLP(
  (layers): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [8]:
multilabel_model = MLP(
        input_size=512,
        activation="leaky_relu",
        dropout=0.2,
        num_classes=20,
        hidden_sizes=[512, 256, 128],
    )
multilabel_model.load_state_dict(
    torch.load("../weights/CT-RATE/model_multilabel.pth")
)
multilabel_model.eval().cuda()

MLP(
  (layers): Sequential(
    (0): Linear(in_features=512, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): LeakyReLU(negative_slope=0.01)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.01)
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=128, out_features=20, bias=True)
  )
)

In [9]:
# 1. Выбери слой, на котором хочешь визуализировать внимание
target_layer = model_lipro.VIT.enc_temporal_transformer  # если self.encode — модуль (Transformer encoder, например)

# 2. Хуки для активаций и градиентов
features = []
gradients = []

def forward_hook(module, input, output):
    features.append(output.detach())

def backward_hook(module, grad_input, grad_output):
    gradients.append(grad_output[0].detach())

target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(backward_hook)


In [ ]:
import torch

inp = input_tensor['tensor'].unsqueeze(0).cuda()
encoded_tokens = model_lipro(inp)
normalized = torch.nn.functional.normalize(encoded_tokens, dim=-1)
out = multilabel_model(normalized)


TEST
torch.Size([1, 24, 24, 24, 512])
TEST1
torch.Size([1, 24, 24, 24, 512])


In [ ]:
from einops import rearrange

score = out[0, 8]  # интересующий класс
score.backward()

acts = features[0]   # [B, T, H, W, D]
grads = gradients[0] # [B, T, H, W, D]


acts_reshaped = rearrange(acts, '(b h w) t d -> b t h w d', b = 1, h = 24, w = 24)
grads_reshaped = rearrange(grads, '(b h w) t d -> b t h w d', b = 1, h = 24, w = 24)

cam = (acts_reshaped * grads_reshaped).mean(dim=-1).relu()
cam.min(), cam.max()


In [21]:
import torch
import torch.nn.functional as F

# cam: [1, T_patch, H_patch, W_patch] -> нужно добавить ось channel
cam_tensor = cam.unsqueeze(0)  # [1, 1, T_patch, H_patch, W_patch]

# целевой размер
T_orig, H_orig, W_orig = 240, 480, 480

cam_upsampled = F.interpolate(
    cam_tensor, size=(T_orig, H_orig, W_orig),
    mode='trilinear', align_corners=False
)

cam_upsampled = cam_upsampled.squeeze()  # [T, H, W]
cam_upsampled = (cam_upsampled - cam_upsampled.min()) / (cam_upsampled.max() + 1e-8)


In [24]:
cam_upsampled.shape

torch.Size([240, 480, 480])

In [23]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import imageio
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

def create_gif(volume, cam, axis, gif_path, cmap_volume='gray', cmap_cam='coolwarm', alpha=0.5):
    """
    volume: np.array или torch.Tensor, shape [T,H,W], значения в [-1,1]
    cam: np.array или torch.Tensor, shape [T,H,W], значения в [0,1]
    axis: 0,1,2 - ось пролистывания
    gif_path: путь для сохранения GIF
    """
    # Преобразуем в numpy
    if isinstance(volume, torch.Tensor):
        volume = volume.cpu().numpy()
    if isinstance(cam, torch.Tensor):
        cam = cam.cpu().numpy()
    
    # Нормализация volume [-1,1] -> [0,1]
    volume = (volume + 1.0) / 2.0
    volume = np.clip(volume, 0, 1)
    
    n_slices = volume.shape[axis]
    images = []
    
    for i in range(n_slices):
        if axis == 0:
            vol_slice = volume[i,:,:]
            cam_slice = cam[i,:,:]
        elif axis == 1:
            vol_slice = volume[:,i,:]
            cam_slice = cam[:,i,:]
        else:
            vol_slice = volume[:,:,i]
            cam_slice = cam[:,:,i]
        
        # Рисуем на Figure
        fig, ax = plt.subplots(figsize=(4,4))
        ax.imshow(vol_slice, cmap=cmap_volume)
        ax.imshow(cam_slice, cmap=cmap_cam, alpha=alpha)
        ax.axis('off')
        plt.tight_layout()

        # Рендерим на canvas и получаем RGBA
        canvas = FigureCanvas(fig)
        canvas.draw()
        buf = np.frombuffer(canvas.buffer_rgba(), dtype=np.uint8)
        buf = buf.reshape(canvas.get_width_height()[::-1] + (4,))
        # конвертируем RGBA -> RGB
        buf = buf[...,:3]
        plt.close(fig)
        images.append(buf)
    
    # Сохраняем GIF
    imageio.mimsave(gif_path, images, duration=0.1)
    print(f"GIF saved to {gif_path}")


# === Пример использования ===
# cam_upsampled: [T,H,W], original_volume: [T,H,W] с значениями [-1,1]
create_gif(input_tensor['tensor'][0], cam_upsampled, axis=0, gif_path='cam_z.gif')  # Z-ось
create_gif(input_tensor['tensor'][0], cam_upsampled, axis=1, gif_path='cam_y.gif')  # Y-ось
create_gif(input_tensor['tensor'][0], cam_upsampled, axis=2, gif_path='cam_x.gif')  # X-ось


GIF saved to cam_z.gif
GIF saved to cam_y.gif
GIF saved to cam_x.gif
